In [0]:
%run ./load_data

Transformation de la table batches

In [0]:
batches_full_list = batches.alias("a").join(
    production_line.alias("b"), 
    col("a.production_line") == col("b.id_plant_production_line"),
    "left").select(
    F.col("b.name").alias("production_line"),
    "a.id_batch",
    "a.batch_number",
    "b.id_plant_production_line",
    "a.planned_datetime",
    F.col("a.created_at").alias("batch_creation_date")).filter(F.col("a.deleted") == False)

Transofrmation des table inferance monitoring

In [0]:
#Récupérer les informations dans la table inference
plant_inference_monitoring = (
    inference_monitoring_candidates_rouen1
    .unionByName(inference_monitoring_candidates_strasbourg2, allowMissingColumns=True)
    .unionByName(inference_monitoring_candidates_prouvy1, allowMissingColumns=True)
    .unionByName(inference_monitoring_candidates_polisy1, allowMissingColumns=True)
    .unionByName(inference_monitoring_candidates_nogent1, allowMissingColumns=True)
    .unionByName(inference_monitoring_candidates_nogent2, allowMissingColumns=True)
    .unionByName(inference_monitoring_candidates_buzau1, allowMissingColumns=True) 
    .unionByName(inference_monitoring_candidates_bolelemi1, allowMissingColumns=True) #modifié
)

In [0]:
parameters_process_activities = (
    parameters_process_activities_st2.
    unionByName(parameters_process_activities_ng1)
    .unionByName(parameters_process_activities_ng2)
    .unionByName(parameters_process_activities_ro1)
    .unionByName(parameters_process_activities_pr1)
    .unionByName(parameters_process_activities_pl1)
    .unionByName(parameters_process_activities_bu1) 
    .unionByName(parameters_process_activities_bo1) #modifié
)

#ici on valide donc les localization où on attend des valeurs (où les valeurs == TRUE dans la table de ref process_activities)
parameters_process_activities_filter = parameters_process_activities[parameters_process_activities["is_target_localization"] == True]


In [0]:
#Renommer correctement les lignes de productions pour éviter les nommeclatures différentes
plant_rename = plant_inference_monitoring.withColumn(
    "pdl",
    when(col("production_line") == "NG2", "NOGENT2")
    .when(col("production_line") == "NG1", "NOGENT1")
    .when(col("production_line") == "ST2", "STRASBOURG2")
    .when(col("production_line") == "ST1", "STRASBOURG1")
    .when(col("production_line") == "RO1", "ROUEN1")
    .when(col("production_line") == "PR1", "PROUYY1")
    .when(col("production_line") == "PO1", "POLISY1")
    .when(col("production_line") == "BU1", "BUZAU1")
    .when(col("production_line") == "BO1", "BOLELEMI1").otherwise(col("production_line"))) #modifié
    

In [0]:
#transformer les target_localisation dans la nommenclature générale pour pouvoir faire les jointures 
plant_rename = plant_rename.withColumn(
  "target_localization",
  F.when(F.col("target_localization") == "steep_c1", "steeping_cycle_1")
  .when(F.col("target_localization") == "steep_c2", "steeping_cycle_2")
  .when(F.col("target_localization") == "kiln_c1", "kilning_cycle_1")
  .when(F.col("target_localization") == "kiln_c2", "kilning_cycle_2")
  .when(F.col("target_localization") == "germ_c1", "germination_cycle_1")
  .when(F.col("target_localization") == "germ_c2", "germination_cycle_2")
  .when(F.col("target_localization") == "germ_c3", "germination_cycle_3")
  .when(F.col("target_localization") == "germ_c4", "germination_cycle_4")
  .when(F.col("target_localization") == "germ_c5", "germination_cycle_5")
  .when(F.col("target_localization") == "pregerm", "pre_germination"))

flaguer les recos où la dernière tentative d'une localisation a une différence de 1h ou + avec la 1ère tentative de reco de la localisation suivante. Cela permettra d'identifier les recommandations qui sont réellement en échec vs là où il y a eu un overlap

In [0]:
#récupérer la liste des localisations où on attend une recommandation par site et ajouter un ordre afin d'identifier les localisations qui se suivent
window_seq = Window.partitionBy("production_line").orderBy("sequence")

parameters_process_activities_with_seq = (
    parameters_process_activities_filter
        .withColumn("sequence_overlap", F.lit(None))
        .withColumn("sequence", F.row_number().over(window_seq))
)

In [0]:
window_spec_overlap_global_ranking = (
    Window
    .partitionBy("batch_id", "target_localization")
    .orderBy(F.col("job_execution_datetetime").asc())
)

df_overlap_global_ranking = (
    plant_rename
    .withColumn("rank_in_batch_localization", F.row_number().over(window_spec_overlap_global_ranking))
)

df_overlap_global_ranking = (
    df_overlap_global_ranking
    .orderBy(F.concat_ws('', F.col("batch_id"), F.col("target_localization")).asc())
)

In [0]:
df_flagged_min_localisation_with_seq = df_overlap_global_ranking.alias("a").join(
    parameters_process_activities_with_seq.alias("b"),
    (F.col("a.production_line") == F.col("b.production_line")) &
    (F.col("a.target_localization") == F.col("b.activity_code")),
    "left"
).select("a.*",
         F.col("b.sequence").alias("sequence_overlap"))



In [0]:
#flaguer les recos où la dernière tentative d'une localisation a une différence de 1h ou + avec la 1ère tentative de reco de la localisation suivante. Cela permettra d'identifier les recommandations qui sont réellement en échec vs là où il y a eu un overlap

window_max_rank_localisation = (
    Window
    .partitionBy("batch_id", "sequence_overlap")
)

df_flagged_max_localisation = (
    df_flagged_min_localisation_with_seq
    .withColumn(
        "max_rank_in_batch_localization",
        F.max("rank_in_batch_localization").over(window_max_rank_localisation)
    )
    .withColumn(
        "is_last_seq_1",
        F.when(
            (F.col("sequence_overlap") == 1) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_last_seq_2",
        F.when(
            (F.col("sequence_overlap") == 2) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_last_seq_3",
        F.when(
            (F.col("sequence_overlap") == 3) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_last_seq_4",
        F.when(
            (F.col("sequence_overlap") == 4) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_last_seq_5",
        F.when(
            (F.col("sequence_overlap") == 5) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_last_seq_6",
        F.when(
            (F.col("sequence_overlap") == 6) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_last_seq_7",
        F.when(
            (F.col("sequence_overlap") == 7) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_last_seq_8",
        F.when(
            (F.col("sequence_overlap") == 8) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_last_seq_9",
        F.when(
            (F.col("sequence_overlap") == 9) &
            (F.col("rank_in_batch_localization") == F.col("max_rank_in_batch_localization")),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
)

df_flagged_min_localisation = (df_flagged_max_localisation
    .withColumn(
        "is_first_seq_2",
        F.when(
            (F.col("sequence_overlap") == 2) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    ) 
    .withColumn(
        "is_first_seq_3",
        F.when(
            (F.col("sequence_overlap") == 3) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )                         
    .withColumn(
        "is_first_seq_4",
        F.when(
            (F.col("sequence_overlap") == 4) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_first_seq_5",
        F.when(
            (F.col("sequence_overlap") == 5) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_first_seq_6",
        F.when(
            (F.col("sequence_overlap") == 6) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_first_seq_7",
        F.when(
            (F.col("sequence_overlap") == 7) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_first_seq_8",
        F.when(
            (F.col("sequence_overlap") == 8) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_first_seq_9",
        F.when(
            (F.col("sequence_overlap") == 9) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    .withColumn(
        "is_first_seq_10",
        F.when(
            (F.col("sequence_overlap") == 10) &
            (F.col("rank_in_batch_localization") == 1),
            F.lit(True)
        ).otherwise(F.lit(False))
    )
    
).orderBy(F.concat_ws('', F.col("batch_id"), F.col("target_localization")).asc())

In [0]:
max_seq = 10 

agg_exprs = []

for n in range(1, max_seq):
    agg_exprs.append(
        F.max(
            F.when(
                F.col(f"is_last_seq_{n}") == True,
                F.col("job_execution_datetetime")
            )
        ).alias(f"end_seq_{n}")
    )
    agg_exprs.append(
        F.min(
            F.when(
                F.col(f"is_first_seq_{n+1}") == True,
                F.col("job_execution_datetetime")
            )
        ).alias(f"start_seq_{n+1}")
    )

df_seq_bounds = (
    df_flagged_min_localisation
    .groupBy("batch_id")
    .agg(*agg_exprs)
)




In [0]:
for n in range(1, max_seq):
    df_seq_bounds = df_seq_bounds.withColumn(
        f"diff_minutes_seq_{n}_{n+1}",
        (
            F.col(f"start_seq_{n+1}").cast("long")
            - F.col(f"end_seq_{n}").cast("long")
        ) / 60
    )

In [0]:
column_flg_overlap = (
    df_flagged_min_localisation_with_seq.alias("a")
    .join(
        df_seq_bounds.alias("b"),
        F.col("a.batch_id") == F.col("b.batch_id"),
        "left"
    )
    .withColumn(
        "overlap",
        F.when(F.col("sequence") == 1, F.col("diff_minutes_seq_1_2"))
         .when(F.col("sequence") == 2, F.col("diff_minutes_seq_2_3"))
         .when(F.col("sequence") == 3, F.col("diff_minutes_seq_3_4"))
         .when(F.col("sequence") == 4, F.col("diff_minutes_seq_4_5"))
         .when(F.col("sequence") == 5, F.col("diff_minutes_seq_5_6"))
         .when(F.col("sequence") == 6, F.col("diff_minutes_seq_6_7"))
         .when(F.col("sequence") == 7, F.col("diff_minutes_seq_7_8"))
         .when(F.col("sequence") == 8, F.col("diff_minutes_seq_8_9"))
         .when(F.col("sequence") == 9, F.col("diff_minutes_seq_9_10"))
         .otherwise(F.lit(None))
    )
).select(
    "a.*",
    "overlap"
)

In [0]:
# Définir la fenêtre pour row_number pour chercher la dernière ligne de chaque localization par batch_number
window_spec_max_date = Window.partitionBy(
    "batch_id",  
    "pdl", 
    "target_localization"
).orderBy(F.col("job_execution_datetetime").desc())

# Appliquer row_number et filtrer pour garder les premières lignes par partition
max_date = column_flg_overlap.filter(
    F.to_date(column_flg_overlap.calculation_interval_after) >= "2024-10-01"  # garder uniquement les valeurs à partir du 1 octobre car les données commencent à ce moment sur table ds
).select(
    F.col("pdl").alias("production_line"), 
    "batch_id", 
    "calculation_interval_after", 
    "job_execution_datetetime", 
    "target_localization", 
    "automatic_missing_values", 
    "manual_missing_values", 
    "is_success", 
    "batch_status",
    "overlap",
    F.row_number().over(window_spec_max_date).alias("rn")  # garder uniquement les lignes de la dernière valeur de job_execution_datetetime
)

# Filtrer pour ne conserver que les lignes où row_number = 1
max_date = max_date.filter(max_date.rn == 1).drop("rn")

Transformation de la table process activities

In [0]:
#on récupère les localizations pour pouvoir identifier les valeurs des target_localizations sur batch_production_planning qui n'a qu'un id localization

activity_mapping = processes_activities.alias("a").join(
    parameters_localizations.alias("b"),
    F.col("a.code") == F.col("b.code"),
    "left").select(
          "a.sequence",
          "a.code",
          "a.id_process_activity",
          "b.id_parameter_localization")


Transformation de batch_planning

In [0]:
# Calcul de la date d'aujourd'hui - 1 jour
yesterday = F.current_date() - F.expr("INTERVAL 0 DAY")

# Jointure et filtrage
batch_planning = (
    batches_production_planning.alias("a")
    .join(activity_mapping.alias("b"), F.col("a.activity") == F.col("b.id_process_activity"), "left")
    .select(
        F.col("a.batch"),
        F.col("a.calculation_interval_after"),
        F.col("b.id_parameter_localization"),
        F.col("b.sequence"),
        F.col("b.code")
    )
    .filter(
        (F.col("a.calculation_interval_after") >= F.lit("2024-09-23 00:00:00").cast("timestamp"))
    )
)


In [0]:
#récupérer a ligne de production des batch pour pouvoir identifier la liste des valeurs attentues par ligne de production
batch_planning = batch_planning.alias("a").join(
    batches.alias("b"), F.col("a.batch") == F.col("b.id_batch"), "left").select(
        F.col("b.production_line"),
        F.col("a.*"),

)

batch_planning = batch_planning.alias("a").join(
    production_line.alias("b"), F.col("a.production_line") == F.col("b.id_plant_production_line"), "left").select(
        F.col("b.name"),
        F.col("a.*")
)

Récupérer la liste des recommandations attendues et donc la liste des target localization où on attend au moins une recommandation IA. pour ça on utilisera Batch_production_planning comme référence des recos attendues.

On se sert également des tables paramters_process_activities pour identifier les target_localizations attendues par ligne de production.Certaines localisations peuvent être flaguées en false dans ces tables pour dire qu'on attend pas particulièrement de recommandation sur une localization d'une ligne de production. 

In [0]:
batch_planning_localizations_attendues = batch_planning.alias("a").join(
    parameters_process_activities_filter.alias("b"),
    (F.col("a.code") == F.col("b.activity_code")) & (F.col("a.name") == F.col("b.production_line")),
    "inner"
).select(
    F.col("a.*")
)

Maintenant qu'on a la liste des recommandations attendues pour chaque batch_id, on va récupérer la liste des recommandations générées. Pour cela on va se servir de la table des recommandations côté pg pour identifier les id_reco non null. 

In [0]:
# Groupement sur la table recommendations
recommendations_grouped = (
    recomendations.groupBy("id_recommendation", "batch", "target_localization", "created_at", "obsolescence_status", "status", "deleted")
    .agg(F.first("id_recommendation").alias("recommendation_id"))
).filter(F.col("deleted") == False)

join_reco_batch_planning = (
    batch_planning_localizations_attendues.alias("a")
    .join(
        recommendations_grouped.alias("r"),
        [
            F.col("a.batch") == F.col("r.batch"),
            F.col("a.id_parameter_localization") == F.col("r.target_localization"),
        ],
        "left"
    )
    .select(
        F.col("a.production_line").alias("id_production_line"),
        F.col("a.batch").alias("batch_id"),
        F.col("a.calculation_interval_after").alias("calculation_interval_after_batch_planning"),
        F.col("id_parameter_localization"),
        F.col("r.recommendation_id").alias("id_recommendation"),
        F.col("r.created_at").alias("date_creation_reco"),
        F.col("r.obsolescence_status").alias("obsolescence_status"),
        F.col("r.status").alias("evaluation_status")
    )
)


In [0]:
# filter les paires batch_id et target_localization avec plusieurs id_reco et garder le plus récent

# Fenêtre pour partitionner par batch_id et id_parameter_localization
window_spec = Window.partitionBy("batch_id", "id_parameter_localization") \
                    .orderBy(F.col("date_creation_reco").desc())

# Ajouter le rang
succes_reco_count_distinct = join_reco_batch_planning.withColumn(
    "row_num", row_number().over(window_spec)
).withColumn(
    "flg_lasted_reco", F.when(F.col("row_num") == 1, F.lit(1)).otherwise(F.lit(0))
)

traitement des prd_cell et workshop

old_stak_localisation_filtered = old_stak_localisation.filter((F.col("production_line") == "NG1") | (F.col("production_line") == "NG2" ) | (F.col("production_line") == "PR1" ))

old_stak_localisation_filtered = old_stak_localisation_filtered.select(
    "batch_id",
    F.col("steep_c1_vessel").alias("steep_vessel1"),
    F.col("steep_c2_vessel").alias("steep_vessel2"),
    F.col("germ_c1_vessel").alias("germ_vessel1"),
    F.col("germ_c2_vessel").alias("germ_vessel2"),
    F.col("germ_c3_vessel").alias("germ_vessel3"),
    F.col("germ_c4_vessel").alias("germ_vessel4"),
    F.col("germ_c5_vessel").alias("germ_vessel5"),
    F.col("kiln_c1_vessel").alias("kiln_vessel1"),
    F.col("kiln_c2_vessel").alias("kiln_vessel2")
)


In [0]:
new_stack_localisation = (
    new_stack_localisation_ro1
    .unionByName(new_stack_localisation_pr1)
    .unionByName(new_stack_localisation_ng1)
    .unionByName(new_stack_localisation_ng2)
    .unionByName(new_stack_localisation_bu1)
    .unionByName(new_stack_localisation_st2)
    .unionByName(new_stack_localisation_po1)
    .unionByName(new_stack_localisation_bo1)) #modifié

In [0]:
localization_events = new_stack_localisation.withColumn(
    "steep_vessel1",
    F.when((F.col("prd_workshop") == "steeping") & (F.col("prd_cell") == 1), "steep_vessel1").otherwise(F.lit(None))
).withColumn(
    "steep_vessel2",
    F.when((F.col("prd_workshop") == "steeping") & (F.col("prd_cell") == 2), "steep_vessel2").otherwise(F.lit(None))
).withColumn(
    "germ_vessel1",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 1), "germ_vessel1").otherwise(F.lit(None))
).withColumn(
    "germ_vessel2",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 2), "germ_vessel2").otherwise(F.lit(None))
).withColumn(
    "germ_vessel3",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 3), "germ_vessel3").otherwise(F.lit(None))
).withColumn(
    "germ_vessel4",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 4), "germ_vessel4").otherwise(F.lit(None))
).withColumn(
    "germ_vessel5",
    F.when((F.col("prd_workshop") == "germination") & (F.col("prd_cell") == 5), "germ_vessel5").otherwise(F.lit(None))
).withColumn(
    "kiln_vessel1",
    F.when((F.col("prd_workshop") == "kilning") & (F.col("prd_cell") == 1), "kiln_vessel1").otherwise(F.lit(None))
).withColumn(
    "kiln_vessel2",
    F.when((F.col("prd_workshop") == "kilning") & (F.col("prd_cell") == 2), "kiln_vessel2").otherwise(F.lit(None))
)



# Agréger chaque colonne par batch_id en prenant la première valeur non nulle (si présence)
df_aggregated = localization_events.groupBy("batch_id").agg(
    first("steep_vessel1", ignorenulls=True).alias("steep_vessel1"),
    first("steep_vessel2", ignorenulls=True).alias("steep_vessel2"),
    first("germ_vessel1", ignorenulls=True).alias("germ_vessel1"),
    first("germ_vessel2", ignorenulls=True).alias("germ_vessel2"),
    first("germ_vessel3", ignorenulls=True).alias("germ_vessel3"),
    first("germ_vessel4", ignorenulls=True).alias("germ_vessel4"),
    first("germ_vessel5", ignorenulls=True).alias("germ_vessel5"),
    first("kiln_vessel1", ignorenulls=True).alias("kiln_vessel1"),
    first("kiln_vessel2", ignorenulls=True).alias("kiln_vessel2")
)


localization_events = df_aggregated

In [0]:
prd_cell_workshop = localization_events #modifié

Concaténation des tables du monitoring DE, des mesures et de l'ascendance

In [0]:
ascendance = (ascendance_st2
              .unionByName(ascendance_pr1, allowMissingColumns=True)
              .unionByName(ascendance_po1, allowMissingColumns=True)
              .unionByName(ascendance_ro1, allowMissingColumns=True)
              .unionByName(ascendance_ng1, allowMissingColumns=True)
              .unionByName(ascendance_ng2, allowMissingColumns=True)
              .unionByName(ascendance_bu1, allowMissingColumns=True)
              .unionByName(ascendance_bo1, allowMissingColumns=True))

ascendance = ascendance.withColumn("prd_line",
                                   F.when(F.col("prd_line") == "ST2", "STRASBOURG2")
                                    .when(F.col("prd_line") == "NG1", "NOGENT1")
                                    .when(F.col("prd_line") == "NG2", "NOGENT2")
                                    .when(F.col("prd_line") == "PO1", "POLISY1")
                                    .when(F.col("prd_line") == "PR1", "PROUVY1")
                                    .when(F.col("prd_line") == "RO1", "ROUEN1")
                                    .when(F.col("prd_line") == "BU1", "BUZAU1")
                                    .when(F.col("prd_line") == "BO1", "BOLELEMI1")
                                   .otherwise(F.col("prd_line"))) #modifié
        
                    

In [0]:
missing_automatic_measures = (missing_automatic_measures_st2
              .unionByName(missing_automatic_measures_pr1, allowMissingColumns=True)
              .unionByName(missing_automatic_measures_po1, allowMissingColumns=True)
              .unionByName(missing_automatic_measures_ro1, allowMissingColumns=True)
              .unionByName(missing_automatic_measures_ng1, allowMissingColumns=True)
              .unionByName(missing_automatic_measures_ng2, allowMissingColumns=True)
              .unionByName(missing_automatic_measures_bu1, allowMissingColumns=True)
              .unionByName(missing_automatic_measures_bo1, allowMissingColumns=True)) #modifié
        

In [0]:
missing_asset_measure = (missing_asset_measure_ng1
              .unionByName(missing_asset_measure_ng2, allowMissingColumns=True)
              .unionByName(missing_asset_measure_pr1, allowMissingColumns=True)
              .unionByName(missing_asset_measure_ro1, allowMissingColumns=True)
              .unionByName(missing_asset_measure_st2, allowMissingColumns=True)
              .unionByName(missing_asset_measure_po1, allowMissingColumns=True)
              .unionByName(missing_asset_measure_bu1, allowMissingColumns=True)
              .unionByName(missing_asset_measure_bo1, allowMissingColumns=True)) #modifié

In [0]:
missing_asset_measure_with_prd_line = missing_asset_measure.alias("a").join(
    batches.alias("b"),
    F.col("a.batch_id") == F.col("b.id_batch"),
    "left").select("a.*",
                   F.col("b.production_line").alias("id_prd_line"))
    
missing_asset_measure_with_prd_line = missing_asset_measure_with_prd_line.alias("a").join(
    production_line.alias("b"),
    F.col("a.id_prd_line") == F.col("b.id_plant_production_line"),
    "left"
).select("a.*",
         (F.col("b.name")).alias("production_line")
)

missing_asset_measure_with_tg_name = missing_asset_measure_with_prd_line.withColumn(
    "target_localization_name",
    F.when(F.col("target_localization") == "steep_c1", "steeping_cycle_1")
    .when(F.col("target_localization") == "steep_c2", "steeping_cycle_2")
    .when(F.col("target_localization") == "kiln_c1", "kilning_cycle_1")
    .when(F.col("target_localization") == "kiln_c2", "kilning_cycle_2")
    .when(F.col("target_localization") == "germ_c1", "germination_cycle_1")
    .when(F.col("target_localization") == "germ_c2", "germination_cycle_2")
    .when(F.col("target_localization") == "germ_c3", "germination_cycle_3")
    .when(F.col("target_localization") == "germ_c4", "germination_cycle_4")
    .when(F.col("target_localization") == "germ_c5", "germination_cycle_5")
    .when(F.col("target_localization") == "pregerm", "pre_germination"))

missing_asset_measure_with_tg_code = missing_asset_measure_with_tg_name.alias("a").join(
    parameters_localizations.alias("b"),
    F.col("a.target_localization_name") == F.col("b.code"),
    "left"
).select("a.*",
         (F.col("b.id_parameter_localization"))
)

missing_asset_measure_with_tg_code = missing_asset_measure_with_tg_code.filter(F.col("is_last_recommendation_successful") == False)

In [0]:
measurement_struct_raw_schema = StructType([
    StructField("value", StringType(), True),
    StructField("status", StringType(), True),
    StructField("computed_value", StringType(), True)
])

def normalize_measurement_pivot(df, production_line_label=None, required_measure_cols=None):
    technical_cols = {
        "prd_line",
        "mes_number",
        "batch_id",
        "created_at",
        "modified_at",
        "flag_data_checking",
        "id",
        "updated_at",
        "deleted",
        "is_deleted",
        "ingestion_date",
        "_rescued_data"
    }

    available_cols = df.columns

    if required_measure_cols is None:
        measure_cols = [c for c in available_cols if c not in technical_cols]
    else:
        measure_cols = [
            c for c in required_measure_cols
            if c in available_cols and c not in technical_cols
        ]

    if len(measure_cols) == 0:
        return spark.createDataFrame([], schema=StructType([
            StructField("prd_line", StringType(), True),
            StructField("batch_id", StringType(), True),
            StructField("measurement_created_at", TimestampType(), True),
            StructField("measure_name", StringType(), True),
            StructField("measure_status", StringType(), True),
            StructField("measure_value", DecimalType(19, 6), True),
            StructField("measure_computed_value", DecimalType(19, 6), True),
        ]))

    df_normalized = df
    dtype_map = dict(df.dtypes)

    for c in measure_cols:
        col_type = dtype_map.get(c)

        if col_type == "string":
            df_normalized = df_normalized.withColumn(
                c,
                F.from_json(F.col(c), measurement_struct_raw_schema)
            )
        else:
            df_normalized = df_normalized.withColumn(
                c,
                F.struct(
                    F.col(f"{c}.value").cast("string").alias("value"),
                    F.col(f"{c}.status").cast("string").alias("status"),
                    F.col(f"{c}.computed_value").cast("string").alias("computed_value")
                )
            )

    kv_structs = ", ".join([f"'{c}', `{c}`" for c in measure_cols])

    select_prd_line = (
        f"'{production_line_label}' as prd_line"
        if production_line_label is not None
        else "prd_line"
    )

    df_long = df_normalized.selectExpr(
        "prd_line",
        "batch_id",
        "created_at",
        "modified_at",
        "flag_data_checking",
        f"stack({len(measure_cols)}, {kv_structs}) as (measure_name, measure_payload)"
    )

    parsed_df = (
        df_long
        .filter(
        (F.col("measure_payload.status").isin("OK", "UNFULFILLED")) &
        (F.col("measure_payload.value").isNotNull())
         )
        .withColumn("measure_status", F.col("measure_payload.status"))
        .withColumn("measure_value", F.col("measure_payload.value").cast(DecimalType(19, 6)))
        .withColumn("measure_computed_value", F.col("measure_payload.computed_value").cast(DecimalType(19, 6)))
    )

    return parsed_df.select(
        "prd_line",
        "batch_id",
        "created_at",
        "modified_at",
        "flag_data_checking",
        "measure_name",
        "measure_status",
        "measure_value",
        "measure_computed_value"
    )

In [0]:
def read_measurement_pivot_asof(
    table_name: str,
    timestamp_asof: str,
    # required_cols: list[str],
    batch_ids: list[str] = None
):
    df = (
        spark.read
        .option("timestampAsOf", timestamp_asof)
        .table(table_name)
    )

    # cols = [c for c in required_cols if c in df.columns]
    # df = df.select(*cols)

    if batch_ids:
        df = df.filter(F.col("batch_id").isin(batch_ids))

    return df